# Principal Component Analysis (PCA)

In this notebook, we use PCA to summarize variation across mineral-composition measurements and inspect how the resulting scores relate to recorded geological metadata. PCA is exploratory: it can reveal patterns in the selected data but cannot by itself identify or establish the petrological processes that produced them.

## Science Question: How do mineral compositions vary with recorded tectonic setting?

Geochemical analyses of minerals are often reported as weight percentages of oxide components (e.g., SIO2, TIO2, Al2O3, FeO, and MgO). These measurements are compositional and may be strongly correlated. PCA expresses selected correlated variables as uncorrelated principal components, providing a lower-dimensional view of variation in the measured data.

PCA can help us:
- visualize many geochemical variables in two or three principal dimensions,
- explore whether analyses associated with different recorded geological contexts occupy different regions of score space, and
- reduce redundancy before further exploratory analysis, such as clustering.

## PCA technique
The goal of PCA is to convert a set of observations into uncorrelated principal components such that the first component explains the maximum variance in the data, the second component explains the maximum variance in the data orthogonal to the first component, and so on.

### Problem Setup

Suppose we have $n$ samples (rocks/minerals in our case), each described by $p$ features (element oxide wt%). We can think of the data as a matrix, $\mathbf{X}$, with dimensions $\{n \times p\}$, where row $i$ is the composition of sample $i$ and column $j$ is the $j$-th oxide across all samples.

First, we center each column by removing its mean, $\bar{x}_j$, such that every feature has zero mean:

$$
\tilde{X}_{ij} = X_{ij} - \bar{x}_j , \qquad \bar{x}_j = \frac{1}{n}\sum_{i=1}^n X_{ij}
$$

The above step is required because PCA looks for directions of maximum variance, and variance is defined relative to the mean. This means that without centering, the first component would simply point toward the mean of the data rather than describing how the data vary around it.

This can be visualized in the following figure.

<img src='./images/PCA_centering.png' width="400" align="center"/>

Image: The principal components, PC1 and PC2, showing the directions of maximum variance in the original (left) and centered data (right). PC1 in the left image is dominated by the mean of the data, without explaining the variability in the data. Image: Ghojogh et al., 2023.

### Finding the principal components

From the centered data, $\mathbf{X_c}$ , we calculate the covariance matrix, $C = \frac{1}{n-1}\mathbf{X_c}^T \mathbf{X_c}$
The covariance matrix describes how the features vary individually and how they vary together.

PCA finds the directions in the data along which the variation is largest. These directions are obtained from the **eigenvectors** of the covariance matrix.

For each eigenvector, $v_k$, $C v_k = \lambda_k v_k$,

where:
- $v_k$ is an eigenvector, which gives the direction of the (k)-th principal component, and
- $\lambda_k$ is the corresponding eigenvalue, which tells us how much variance is present along that direction.

Assuming that $\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_p$, the first principal component is the eigenvector corresponding to the largest eigenvalue, the second principal component is the eigenvector corresponding to the next-largest variance (i.e., second largest eigenvalue), and so on.

### Projecting the data onto the principal components

Once the eigenvectors, $v_1, v_2, \ldots, v_m$, have been found, we can project the centered data onto them.

For the nth principal component, $z_n = X_c v_n$, or in matrix form, $\text{Z} = \text{X}_c \text{V}$.

Here:
 - V is the matrix of principal-component directions (eigenvectors) of size $\{p \times m\}$,  where $m$ is the number of principal components we want to keep, usually $m < p$ since we want to reduce the dimensionality of the data.
 - Z is the matrix that represents the data projected onto the principal components and is of size $\{n \times m\}$.

 In other words, we reduce the size of the data from $n \times p$ to $n \times m$ by projecting the data onto the first $m$ principal components.

## PCA in python

In this notebook we will use `pandas` and `scikit-learn` to perform Principal Component Analysis (PCA) on mineral-composition data downloaded from [PetDB](https://www.earthchem.org/petdb). The samples include also information about the associated rock type and its tectonic setting.

We will:

- inspect the input data and identify reported oxide measurements,
- select commonly reported oxide features and retain rows with complete measurements for those features,
- standardize the features and fit PCA using `sklearn.decomposition.PCA`,
- visualize explained variance using scores colored by tectonic setting, and
- compute oxide loadings to investigate the contribution of each oxide in the PC

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (6, 5)

### 1. Inspect the data

The raw PetDB export contains one row per analyzed sample, along with metadata columns (sample name, location, rock type, etc.) and, for every possible oxide, a value column (`<oxide> (wt%)`) plus other columns describing how it was measured. For our analysis, we first need  to isolate just the oxide value columns.

In [ ]:
# Load the raw PetDB export and investigate the columns.
# Some data is string while some is numeric, so we use the `low_memory=False` option to avoid dtype warnings
# This is memory intensive, especially for large datasets, so if you run into memory issues, remove this option

input_geochem_data = pd.read_csv('data/earthchem_download_pca.txt', sep="\t", low_memory=False)
print(f"Raw data: {input_geochem_data.shape[0]} samples, {input_geochem_data.shape[1]} columns")
input_geochem_data.columns

In [ ]:
# oxide-like names: SIO2, AL2O3, FEO, FEOT, FE2O3T, NA2O, K2O, P2O5, ZRO2 ...
ox = input_geochem_data.filter(regex=r"^[A-Z]{1,2}\d?O\d?T?$")

# remove volatiles/gases that the pattern also catches
ox = ox.drop(columns=["H2O", "CO2"], errors="ignore")

ox = ox.apply(pd.to_numeric, errors="coerce")
print(list(ox.columns))

### 2. Clean data: Select dominant oxide features

We identify oxide-like columns in the preceding cell and from it, select the 10 most frequently reported features.

> An important thing to note here is that a missing measurement (or a NaN value) does not mean that the targeted oxide is absent (or has a zero value). Geochemical analyses varies by the technique or the lab in which the measurement is made. Therefore, treating a missing oxide measurement as 0 might cluster samples based on the lab or method producing the measurements, rather than chemistry.

To avoid this, we retain only rows with measurements for all selected features before fitting PCA. This complete-case choice would reduce the dataset size.

We keep `ROCK TYPE` and `TECTONIC SETTING` for visualizing the PCA scores. These describe the database's geological context and should not be treated as labels validated independently of the geochemical measurements.

In [ ]:
n_most_abundant_oxides = 10
counts = ox.notna().sum()
feature_names = counts.nlargest(n_most_abundant_oxides).index

print("Fraction of samples with measurements for the selected features: \n",
      counts[feature_names]/counts.max())

# Retain only rows with complete measurements for the selected features
analysis_df = (ox[feature_names].join(input_geochem_data[["ROCK TYPE", "TECTONIC SETTING"]]).dropna(subset=feature_names))

### 3. Preprocessing: standardization

As we discussed above, we center each selected oxide feature so it has zero mean so that PCA does not simply point toward the mean of the data. Additionally, we divide each feature by its standard deviation so that the features are scaled to unit variance. This is important because oxide features are measured in different ranges (e.g., SIO2 is often 40-70 wt% while TIO2 is often <1 wt%). If we do not scale the features, PCA will be dominated by the variation of SIO2.

The standardization is done using the `StandardScaler` from `sklearn.preprocessing`.

In [ ]:
X = analysis_df[feature_names].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Input matrix shape: {X.shape}")
print("Feature means after standardization:", np.mean(X_scaled, axis=0).round(3))
print("Feature standard deviations after standardization:", np.std(X_scaled, axis=0).round(3))

### 4. PCA fitting

We fit PCA using all selected oxide features and based on the explained variance of each component, decide how many components are useful for visualization and further analysis.

In [ ]:
pca = PCA()
scores = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

n_components = len(explained_var)
pcs = np.arange(1, n_components + 1)

fig, ax1 = plt.subplots()
ax1.bar(pcs, explained_var, label="Explained variance")
ax1.set_xlabel("Principal component")
ax1.set_ylabel("Explained variance ratio")
ax1.set_xticks(pcs)

ax2 = ax1.twinx()
ax2.plot(pcs, cumulative_var, color="black", marker="o", label="Cumulative variance")
ax2.set_ylabel("Cumulative explained variance")
ax2.set_ylim(0, 1.05)
ax2.axhline(0.8, color="gray", linestyle="--", linewidth=1, label="80% variance")

fig.legend(loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=3)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

The above plot shows that the first two principal components explain about 50% of the variance in the data. Therefore, we will retain only the first two components for further analysis and visualization.

### 5. Understanding PC scores by tectonic setting

Each point in the PC1-PC2 score plot represents one mineral analysis, colored by its recorded tectonic setting. The plotted setting is directly from the database, not a prediction by PCA. Through this visualization, we can explore how samples from different recorded geological contexts form clusters or occupy different regions of score space.

In [ ]:
labels = analysis_df["TECTONIC SETTING"].astype(str).str.strip()
settings = labels.value_counts().index

fig, ax = plt.subplots(figsize=(8, 4))

for setting in settings:
    mask = (labels == setting).to_numpy()
    ax.scatter(scores[mask, 0], scores[mask, 1], s=20, alpha=0.55, label=setting)

ax.set_xlabel(f"PC1 ({explained_var[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_var[1] * 100:.1f}% variance)")
ax.set_title("PCA scores by recorded tectonic setting")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.grid(alpha=0.3)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8, title="Tectonic setting")
fig.tight_layout(rect=[0, 0, 0.76, 1])
plt.show()

The plot above shows following observations:
- Even though different tectonic settings form clusters in the PC1-PC2 score space, there is a considerable overlap among them. This suggests that the first two principal components do not cleanly separate the samples based on tectonic setting.
- The spread of points for the spreading centers is larger than the other settings, which could either reflect a wider range in the compositional heterogeneity of the samples (due to different partial melting processes, for example) or a data quality issue.

Next, we will inspect the oxide loadings to see which measured features contribute to PC1 and PC2.

### 6. Loadings of each oxide on PC1 and PC2

The **loadings** describe how each oxide relates to each principal component. They are computed as : $$ L = V \Lambda^{1/2} $$ where $V$ is the matrix of eigenvectors and $\Lambda$ is the diagonal matrix of eigenvalues.

We use bar plots to visualize the loadings of each oxide on PC1 and PC2. In these plots, the length of each bar indicates the magnitude of the loading, while the direction (positive or negative) indicates whether the oxide is positively or negatively correlated with the principal component.

In [ ]:
# Correlation loadings for PCA on standardized features.
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharex=True)
for component, ax in enumerate((ax1, ax2)):
    values = loadings[:, component]
    colors = np.where(values >= 0, "steelblue", "indianred")
    ax.barh(feature_names, values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"PC{component + 1} ({explained_var[component] * 100:.1f}% variance)")
    ax.set_xlabel("Correlation loading")
    ax.grid(axis="x", alpha=0.3)

ax1.set_ylabel("Oxide feature")
fig.suptitle("Oxide loadings on the first two principal components")
fig.tight_layout()
plt.show()

The loadings plots show that:
- PC1 is dominated by silica- and calcium-rich oxides, and depleted in iron- and magnesium-rich oxides, and could be indicative of a felsic rocks composition.
- PC2 is mostly dominated in MGO and depleted in Al- and Ca-oxides, and could be indicative of an ultramafic rocks composition.

## Try it yourself:

- Plot the PCA scores colored by rock type instead of tectonic setting. Do the different rock types form clusters in the PC1-PC2 score space?

- Download data from the PetDB Database (www.earthchem.org/petdb) for a different rock type and repeat the PCA analysis. Do you see any interesting patterns?

- What happens if you do not standardize the features before fitting PCA? How does this affect the explained variance and the loadings?

### References
- Ghojogh, B., Crowley, M., Karray, F., Ghodsi, A. (2023). Principal Component Analysis. In: Elements of Dimensionality Reduction and Manifold Learning. [Springer](https://doi.org/10.1007/978-3-031-10602-6_5)
- The data were downloaded from the PetDB Database (www.earthchem.org/petdb) using the following parameters: tectonic setting = Intraplate craton, spreading center, and forearc basins.

### Further Reading
- PCA can also be computed using **singular value decomposition (SVD)**. For the centered data matrix, SVD gives $$\mathbf{X}= \mathbf{U}\mathbf{\Sigma}\mathbf{V}^\mathsf{T}$$ The columns of $\mathbf{V}$ are the principal-component directions, and $\mathbf{U}\mathbf{\Sigma}$ gives the scores.
- Singular values in $\mathbf{\Sigma}$ are nonnegative. For a centered data, the square of singular values divided by $n-1$ are the eigenvalues of the covariance matrix: $\lambda_i = \sigma_i^2/(n-1)$. Eigenvalues are defined for square matrices, and hence we generate a covariance matrix, $\mathbf{X_c}^T \mathbf{X_c}$. The eigenvalues of the covariance matrix are nonnegative.
- SVD works directly on the data matrix, so PCA can be computed without explicitly forming its covariance matrix.

You can read more about SVD decomposition [here](https://math.mit.edu/classes/18.095/2016IAP/lec2/SVD_Notes.pdf).

 &nbsp;<div style="text-align: right">  
    &rarr; <b>NEXT: [t-SNE on geochemical data](./2_manifold_learning.ipynb) </b> <a href=""></a> &nbsp;&nbsp;
     <img src="../../../assets/education-gem-notebooks_icon.png" alt="icon"  style="width:4%">
  </div>